Real-Time Credit Risk Simulation

Lightweight, real-time credit risk analysis pipeline that simulates event ingestion, processes streaming data in pandas, applies logistic regression for prediction, and generates actionable business recommendations.

Sets up the environment and imports `run_pipeline()`.

Checks if the output parquet file exists and deletes it to ensure a fresh pipeline run.

Prints a message to indicate whether the file was removed or not.


In [1]:
import sys
import os
import pandas as pd
import threading
import time
from pathlib import Path
from IPython.display import clear_output

root = "/Users/phillipsmith/Desktop/pythonProjects/real-time-credit-risk-simulation"
sys.path.append(root)

from src.pipeline import run_pipeline

file_path = Path('/Users/phillipsmith/Desktop/pythonProjects/real-time-credit-risk-simulation/src/data/processed/processed_df.parquet')

"""Delete existing parquet if it exists for a fresh execution."""
if file_path.exists():
    file_path.unlink()
    print(f"Deleted existing file: {file_path}", flush=True)
else:
    print("No existing parquet file found. Continuing...", flush=True)

No existing parquet file found. Continuing...


Runs `run_pipeline()` in a separate thread so the notebook can monitor progress in real time.

Continuously reads the output parquet file and prints the DataFrame shape whenever new events are added.

After the pipeline finishes, it prints the final DataFrame size.

In [2]:
"""Run thread on run_pipeline to concurrently produce and consume events"""
pipeline_thread = threading.Thread(target=run_pipeline)
pipeline_thread.start()
print("Monitoring parquet file for updates...\n")

previous_shape = (0, 0)
while pipeline_thread.is_alive():
    if file_path.exists():
        try:
            df = pd.read_parquet(file_path, engine='fastparquet')
            if df.shape != previous_shape:
                print(f"Current DataFrame shape: {df.shape}\n")
                previous_shape = df.shape
        except Exception:
            pass
    time.sleep(0.)

# After pipeline finishes
pipeline_thread.join()
if file_path.exists():
    df = pd.read_parquet(file_path, engine='fastparquet')
    print(f"\nPipeline complete. Final DataFrame shape: {df.shape}")
else:
    print("Pipeline finished but file was not found.")

Monitoring parquet file for updates...

Event producing has initiated...
Event consuming has initiated...

Consumed batch of 10000 events. Total events processed: 10000

Current DataFrame shape: (10000, 8)

Consumed batch of 10000 events. Total events processed: 20000

Current DataFrame shape: (20000, 8)

Consumed batch of 10000 events. Total events processed: 30000

Current DataFrame shape: (30000, 8)

Consumed batch of 10000 events. Total events processed: 40000

Current DataFrame shape: (40000, 8)

All events have been produced...

Consumed batch of 5063 events. Total events processed: 45063

Simulation Complete. Processed Dataframe shape: (45063, 8)
Current DataFrame shape: (45063, 8)


Pipeline complete. Final DataFrame shape: (45063, 8)
